# NB_Gold_Model
**PNG Masalais (fictional club, synthetic data)**. Silver to Gold: the star schema the Direct Lake model reads.

Run with **Run all**. Default lakehouse: **LH_Silver**. Writes go to **LH_Gold**. All lakehouses are without schemas.

Design rules:
- **Star schema**: facts hold numbers and keys; dimensions hold descriptions. Every fact links straight to its dimensions
  (no chains like fact to session to programme), because Power BI filters flow best one hop from a dimension.
- **Business logic decided once, here**: agreement-currency amounts, overdue milestones, before/after programme flags.
  The report then only adds up columns, so every page gives the same answer.
- `REPORT_DATE` is the board pack's "as at" date. Change it in one place and every flag follows.

## 1. Setup

In [ ]:
from pyspark.sql import functions as F, Window
from datetime import date

SILVER = "LH_Silver"
GOLD = "LH_Gold"
REPORT_DATE = date(2026, 10, 31)

def rd(t):
    return spark.read.table(f"{SILVER}.{t}")

def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.{name}")
    print(f"{name:<24} {df.count():>7,} rows")

## 2. dim_date
One row per day, 2025 to 2027. Two financial years side by side:
- **AU FY** runs July to June. July 2025 is in `FY2025-26`.
- **PNG FY** is the calendar year.

`AU_FY_Sort` and `MonthSort` exist so Power BI sorts these text columns in time order, not alphabetically.
`IsActual` marks days up to the reporting date, so budget-vs-actual pages can compare like with like.

In [ ]:
terms = [(2025,1,"2025-02-03","2025-04-11"),(2025,2,"2025-04-28","2025-06-27"),(2025,3,"2025-07-14","2025-09-19"),
         (2025,4,"2025-10-06","2025-12-05"),(2026,1,"2026-02-02","2026-04-10"),(2026,2,"2026-04-27","2026-06-26"),
         (2026,3,"2026-07-13","2026-09-18"),(2026,4,"2026-10-05","2026-12-04")]
term_df = spark.createDataFrame(terms, ["TY", "TN", "TS", "TE"]) \
    .withColumn("TS", F.to_date("TS")).withColumn("TE", F.to_date("TE"))

d = spark.sql("SELECT explode(sequence(to_date('2025-01-01'), to_date('2027-12-31'), interval 1 day)) AS Date")
au_start = F.when(F.month("Date") >= 7, F.year("Date")).otherwise(F.year("Date") - 1)
d = (d.withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
      .withColumn("Year", F.year("Date"))
      .withColumn("MonthStart", F.trunc("Date", "month"))
      .withColumn("MonthName", F.date_format("Date", "MMM yyyy"))
      .withColumn("MonthSort", F.year("Date") * 100 + F.month("Date"))
      .withColumn("AU_FY", F.concat(F.lit("FY"), au_start.cast("string"), F.lit("-"),
                                    F.substring((au_start + 1).cast("string"), 3, 2)))
      .withColumn("AU_FY_Sort", au_start)
      .withColumn("PNG_FY", F.year("Date"))
      .withColumn("IsActual", F.col("Date") <= F.lit(REPORT_DATE))
      .withColumn("IsEstablishmentPeriod", (F.col("Date") >= F.lit(date(2025, 7, 1))) & (F.col("Date") <= F.lit(REPORT_DATE))))
d = (d.join(term_df, (d.Date >= term_df.TS) & (d.Date <= term_df.TE), "left")
      .withColumn("SchoolTerm", F.when(F.col("TN").isNotNull(), F.concat_ws(" ", F.col("TY").cast("string"),
                                                                            F.concat(F.lit("T"), F.col("TN").cast("string"))))
                                  .otherwise(F.lit("Holidays")))
      .drop("TY", "TN", "TS", "TE"))
save(d, "dim_date")

## 3. Dimensions
`dim_agreement` folds in the funder's name and type. A funder only matters through its agreements, so one table
(instead of funder to agreement to fact) keeps the model a clean star.

`dim_programme` gets an `UNALLOCATED` row and `dim_account` gets the suspense account 5999, so problem GL lines
still have somewhere to belong. Money is never dropped: it shows up as its own, visible category to chase.

In [ ]:
agr = rd("agreements").join(rd("funders"), "FunderID", "left").select(
    "AgreementID", "AgreementName", "FunderID", "FunderName", "FunderType", "Country", "Currency",
    F.col("CommittedAmount").cast("double").alias("CommittedAmount"),
    F.to_date("StartDate").alias("StartDate"), F.to_date("EndDate").alias("EndDate"),
    "Restriction", "RestrictionNotes", "PaymentBasis")
save(agr, "dim_agreement")

prog = rd("programmes").select("ProgrammeID", "ProgrammeName", "Stream", "IsParticipantBased")
prog = prog.unionByName(spark.createDataFrame(
    [("UNALLOCATED", "Unallocated (missing cost centre)", "Data quality", "No")], prog.columns))
save(prog, "dim_programme")

acct_dim = rd("accounts").unionByName(spark.createDataFrame(
    [("5999", "Suspense - Unallocated", "Expense", "Unclassified (suspense)")],
    ["AccountCode", "AccountName", "AccountType", "CostCategory"]))
save(acct_dim, "dim_account")
save(rd("provinces"), "dim_province")
save(rd("schools").select("SchoolID", "SchoolName", "Level", "ProvinceCode", "IsProgrammeSchool",
                          "ProgrammeJoinDate")
     .withColumn("SchoolGroup", F.when(F.col("IsProgrammeSchool") == "Y", "Programme schools")
                                 .otherwise("Comparison schools")), "dim_school")
save(rd("participants").select("ParticipantID", "GenderLabel", "AgeBand", "AgeAtRegistration",
                               "ProvinceCode", "RegisteredDate", "HomeProgrammeID")
     .withColumn("AgeBandSort", F.when(F.col("AgeBand") == "Under 12", 1).when(F.col("AgeBand") == "12-15", 2)
                                 .when(F.col("AgeBand") == "16-18", 3).when(F.col("AgeBand") == "19-24", 4)
                                 .otherwise(5)), "dim_participant")

## 4. dim_milestone and fact_milestone_update
The update table keeps **every** status update (an audit trail). From it we work out, per milestone:
- latest status and % complete
- completion date (first update marked Complete)
- **IsOverdue** as at the reporting date: due date passed and not complete. This is what Activator will watch.
- days late (for completed milestones, how late they finished; for overdue ones, how late they are now)

In [ ]:
upd = rd("milestone_updates").select(
    "UpdateID", "MilestoneID", F.to_date("UpdateDate").alias("UpdateDate"), "Status",
    F.col("PctComplete").cast("double").alias("PctComplete"), "Comment", "UpdatedBy")
upd = upd.filter(F.col("UpdateDate") <= F.lit(REPORT_DATE))
w = Window.partitionBy("MilestoneID").orderBy(F.col("UpdateDate").desc(), F.col("UpdateID").desc())
upd = upd.withColumn("IsLatest", F.row_number().over(w) == 1)
save(upd, "fact_milestone_update")

latest = upd.filter("IsLatest").select("MilestoneID", F.col("Status").alias("LatestStatus"),
                                       F.col("PctComplete").alias("LatestPctComplete"),
                                       F.col("UpdateDate").alias("LastUpdated"), F.col("Comment").alias("LatestComment"))
done = upd.filter("Status = 'Complete'").groupBy("MilestoneID").agg(F.min("UpdateDate").alias("CompletedDate"))

ms = (rd("milestones").select("MilestoneID", "AgreementID", "ProgrammeID", "Description",
                              F.to_date("DueDate").alias("DueDate"),
                              F.col("TrancheAmount").cast("double").alias("TrancheAmount"), "MilestoneType")
      .join(latest, "MilestoneID", "left").join(done, "MilestoneID", "left"))
rep = F.lit(REPORT_DATE)
ms = (ms.withColumn("LatestStatus", F.coalesce("LatestStatus", F.lit("Not started")))
        .withColumn("IsComplete", F.col("CompletedDate").isNotNull())
        .withColumn("IsOverdue", (~F.col("IsComplete")) & (F.col("DueDate") < rep))
        .withColumn("CompletedLate", F.col("IsComplete") & (F.col("CompletedDate") > F.col("DueDate")))
        .withColumn("DaysLate", F.when(F.col("IsComplete"), F.greatest(F.datediff("CompletedDate", "DueDate"), F.lit(0)))
                                 .when(F.col("IsOverdue"), F.datediff(rep, "DueDate")).otherwise(F.lit(0)))
        .withColumn("RAG", F.when(F.col("IsOverdue"), "Red")
                            .when(F.col("IsComplete"), "Complete")
                            .when(F.col("LatestStatus") == "At risk", "Amber").otherwise("Green")))
save(ms, "dim_milestone")

## 5. fact_gl
Adds **AmountAgreementCcy**: each line in the currency of the agreement that funded it.
- AUD agreement: use the AUD amount (PGK spending in PNG converted at that month's rate)
- PGK agreement: use the PGK amount as booked
- No agreement (bank interest): left blank

Also adds `BudgetLineKey` so actuals line up with budget at Agreement, Programme and Cost category level.
Sign convention stays as the ledger has it: **expenses positive, income negative**. Measures flip income for display.

In [ ]:
acct = spark.read.table(f"{GOLD}.dim_account").select("AccountCode", "AccountType", "CostCategory")
cur = rd("agreements").select("AgreementID", F.col("Currency").alias("AgreementCurrency"))
gl = (rd("gl").join(acct, "AccountCode", "left").join(cur, "AgreementID", "left")
      .withColumn("AmountAgreementCcy",
                  F.when(F.col("AgreementCurrency") == "AUD", F.col("AmountAUD"))
                   .when(F.col("AgreementCurrency") == "PGK", F.col("AmountLocal")))
      .withColumn("DateKey", F.date_format("TxnDate", "yyyyMMdd").cast("int"))
      .withColumn("BudgetLineKey", F.when(F.col("AccountType") == "Expense",
                  F.concat_ws("|", "AgreementID", "ProgrammeID", "CostCategory"))))
save(gl.select("DateKey", "TxnDate", "Entity", "JournalRef", "AccountCode", "ProgrammeID", "AgreementID",
               "BudgetLineKey", "CostCategory", "Currency", "AmountLocal", "AUDtoPGK", "AmountAUD", "AgreementCurrency",
               "AmountAgreementCcy", "Narration", "SourceFile", "DQ_Flag"), "fact_gl")

## 6. fact_budget
Budget is already in agreement currency. We add the AUD equivalent (at the month's rate, for months that have one)
and the same `BudgetLineKey` as the GL.

In [ ]:
fx = rd("fx_rates")
b = (rd("budget").join(fx, "MonthStart", "left")
     .withColumn("BudgetAmount", F.col("BudgetAmount").cast("double"))
     .withColumn("BudgetAUD", F.when(F.col("Currency") == "AUD", F.col("BudgetAmount"))
                               .otherwise(F.round(F.col("BudgetAmount") / F.col("AUDtoPGK"), 2)))
     .withColumn("DateKey", F.date_format("MonthStart", "yyyyMMdd").cast("int"))
     .withColumn("BudgetLineKey", F.concat_ws("|", "AgreementID", "ProgrammeID", "CostCategory")))
save(b.select("DateKey", "MonthStart", "AgreementID", "ProgrammeID", "CostCategory", "BudgetLineKey",
              "Currency", "BudgetAmount", "BudgetAUD"), "fact_budget")

# Shared cost category dimension: GL knows category through the account, budget only at category level.
# One shared table lets a single slicer filter both facts.
cats = (spark.read.table(f"{GOLD}.dim_account").filter("AccountType = 'Expense'").select("CostCategory")
        .union(spark.read.table(f"{GOLD}.fact_budget").select("CostCategory")).distinct())
save(cats, "dim_cost_category")

## 7. Programme facts
`fact_attendance` copies the session's date, programme, province and school onto every attendance row.
That repeats a little data, but it means attendance links **directly** to every dimension. Reach (distinct people)
can then be sliced by province or programme without going through the session table.

In [ ]:
ses = rd("sessions").withColumn("DateKey", F.date_format("SessionDate", "yyyyMMdd").cast("int"))
save(ses.select("SessionID", "DateKey", "SessionDate", "ProgrammeID", "ProvinceCode", "SchoolID",
                "FacilitatorID", "DurationMins"), "fact_session")

att = rd("attendance").join(ses.select("SessionID", "DateKey", "SessionDate", "ProgrammeID", "ProvinceCode", "SchoolID"),
                            "SessionID")
save(att, "fact_attendance")

sa = (rd("school_attendance").join(rd("schools").select("SchoolID", "IsProgrammeSchool", "ProgrammeJoinDate"), "SchoolID")
      .withColumn("DateKey", F.date_format("TermEnd", "yyyyMMdd").cast("int"))
      .withColumn("TermLabel", F.concat_ws(" ", F.col("Year").cast("string"), F.concat(F.lit("T"), F.col("Term").cast("string"))))
      .withColumn("TermSort", F.col("Year") * 10 + F.col("Term"))
      .withColumn("Period", F.when(F.col("IsProgrammeSchool") == "N", "Comparison school")
                             .when(F.col("TermStart") >= F.date_sub("ProgrammeJoinDate", 7), "After joining")
                             .otherwise("Before joining"))
      .withColumn("PeriodSort", F.when(F.col("Period") == "Before joining", 1)
                                 .when(F.col("Period") == "After joining", 2).otherwise(3)))
save(sa.select("SchoolID", "DateKey", "Year", "Term", "TermLabel", "TermSort", "TermStart", "TermEnd",
               "Enrolled", "AvgAttendanceRate", "IsProgrammeSchool", "Period", "PeriodSort"), "fact_school_attendance")

## 8. dq_log (latest run per source table)
Silver keeps every run's log. The report only needs the latest findings.

In [ ]:
dq = rd("dq_log")
w = Window.partitionBy("SourceTable")
dq = dq.withColumn("_latest", F.max("RunId").over(w)).filter(F.col("RunId") == F.col("_latest")).drop("_latest")
save(dq, "dq_log")

## 9. Checks against the answer key

In [ ]:
g = lambda t: spark.read.table(f"{GOLD}.{t}")
print("Funding received vs spent, agreement currency, to 31 Oct 2026:")
fg = g("fact_gl").join(g("dim_account").select("AccountCode", "AccountType"), "AccountCode")
display(fg.filter(F.col("AgreementID").isNotNull()).groupBy("AgreementID")
          .agg(F.round(-F.sum(F.when(F.col("AccountType") == "Revenue", F.col("AmountAgreementCcy"))), 0).alias("Received"),
               F.round(F.sum(F.when(F.col("AccountType") == "Expense", F.col("AmountAgreementCcy"))), 0).alias("Spent"))
          .orderBy("AgreementID"))

print("Budget lines more than 10% over, to date:")
act = fg.filter("AccountType = 'Expense' AND BudgetLineKey IS NOT NULL").groupBy("BudgetLineKey") \
        .agg(F.sum("AmountAgreementCcy").alias("Actual"))
bud = g("fact_budget").filter(F.col("MonthStart") <= F.lit(REPORT_DATE)).groupBy("BudgetLineKey") \
        .agg(F.sum("BudgetAmount").alias("Budget"))
display(bud.join(act, "BudgetLineKey", "left").withColumn("VarPct", F.round(F.col("Actual") / F.col("Budget") - 1, 3))
           .filter("VarPct > 0.10").orderBy(F.desc("VarPct")))

print("Overdue milestones as at", REPORT_DATE)
display(g("dim_milestone").filter("IsOverdue").select("MilestoneID", "Description", "DueDate", "LatestStatus",
                                                      "LatestPctComplete", "DaysLate"))